# Phase 2 — LLM 클라이언트 통합 (캐싱 흡수) 데모

Phase 1 의 시스템 프롬프트 정적/동적 분리 (`__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__` + `get_static_hash`) 를 실제 LLM 호출까지 연결하면서 **두 provider (Gemini / Anthropic) 의 캐싱을 통합 인터페이스로 흡수** 한다.

## 작업순서
1. **Setup** — sys.path patch + import + 환경 변수 안내
2. **1부: 베이스 사용 흐름** — `split_at_boundary` 동작 + GeminiClient 단순 호출 + cache hit 시연
3. **2부: provider 교체** — AnthropicClient 호출 (cache_control marker) + LLMClient Protocol 로 도메인 어댑터 끼우기
4. **3부: escape hatch + 위험 가드** — CachePolicy 컨트롤 + 메트릭 observer + force_invalidate semantic 차이
5. **Cleanup** — registry/metrics 격리 안내
6. **다른 모듈 연계** — 시스템 프롬프트 모듈의 `get_static_hash` 가 본 모듈 캐시 키 source
7. **실습 4개**

## Setup

`uv` 가 정식 ipykernel 등록 전까지 `sys.path` patch 로 임시. 정식 등록은 `uv add --dev jupyter ipykernel && uv run python -m ipykernel install --user --name best-agent-base` (사용자 승인 필요).

**환경 변수**: 실제 호출 시 `.env` 에 `GOOGLE_API_KEY` / `ANTHROPIC_API_KEY` 필요. 본 노트북은 LLM 호출 셀을 주석 처리 — Mock 또는 직접 실행은 사용자 선택.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from best_agent_base.llm.cache_metrics import CacheEvent, CacheMetrics
from best_agent_base.llm.cache_policy import CachePolicy
from best_agent_base.llm.client import LLMClient, LLMResponse, TokenUsage
from best_agent_base.llm.messages import split_at_boundary
from best_agent_base.prompts.render import RenderContext, get_static_hash, render

print('imports OK')

## 1부: 베이스 사용 흐름

Phase 1 의 `render(ctx)` 출력을 보고, 본 모듈의 `split_at_boundary` 가 BOUNDARY 기준으로 어떻게 (static, dynamic) 으로 split 하는지 확인. provider-agnostic 헬퍼 — Gemini / Anthropic 어댑터 모두 재사용.

In [ ]:
ctx = RenderContext()
static, dynamic = split_at_boundary(ctx)
print(f'static len = {len(static)}')
print(f'dynamic len = {len(dynamic)}  (베이스에는 동적부 없음 → 빈 문자열)')
print(f'\nstatic_hash = {get_static_hash(ctx)}  ← 캐시 키 source of truth')

### Gemini 단일 호출 (실제 호출 — 주석 해제 시 GOOGLE_API_KEY 필요)

두 번째 호출부터 `cache_hit=True` — Gemini `CachedContent` 가 정적부를 자동 재사용.

In [ ]:
import asyncio

from dotenv import load_dotenv
from best_agent_base.llm.gemini import GeminiClient

# load_dotenv()
# client = GeminiClient()
# resp = await client.generate(RenderContext())
# print(f'text = {resp.text!r}')
# print(f'cache_hit={resp.cache_hit} static_hash={resp.static_hash}')
# print(f'usage = input={resp.usage.input_tokens} output={resp.usage.output_tokens} cached={resp.usage.cached_tokens}')

# 두 번째 호출
# resp2 = await client.generate(RenderContext())
# print(f'\n[2nd call] cache_hit={resp2.cache_hit}  ← True 기대')

print('주석 해제 후 실행. 실제 호출 시 GOOGLE_API_KEY 필요.')

## 2부: provider 교체 — Open/Closed

동일한 `LLMClient` Protocol 적합 → 호출 코드 1줄만 변경. Anthropic 은 `cache_control: {"type": "ephemeral"}` marker 부착으로 server-side 캐싱.

### 2.1 Anthropic 단일 호출 (실제 호출 — 주석 해제 시 ANTHROPIC_API_KEY 필요)

동일한 `await client.generate(ctx)` 시그니처 — provider 차이 모름. 두 번째 호출부터 server-side cache hit (`usage.cache_read_input_tokens > 0` → `LLMResponse.cache_hit=True`).

In [ ]:
from best_agent_base.llm.anthropic import AnthropicClient

# load_dotenv()
# client_a = AnthropicClient()  # 디폴트 model=claude-sonnet-4-5-20250929
# resp = await client_a.generate(RenderContext())
# print(f'text = {resp.text!r}')
# print(f'cache_hit={resp.cache_hit} static_hash={resp.static_hash}')
# print(f'usage = input={resp.usage.input_tokens} output={resp.usage.output_tokens} cached={resp.usage.cached_tokens}')

# 두 번째 호출 (server-side hash 매칭으로 cache hit)
# resp2 = await client_a.generate(RenderContext())
# print(f'\n[2nd call] cache_hit={resp2.cache_hit}  ← True 기대 (cache_read_input_tokens > 0)')

# 다른 model 로 인스턴스화 (예: Sonnet 대신 Haiku)
# client_haiku = AnthropicClient(model='claude-haiku-4-5-20251001', max_tokens=2048)

print('주석 해제 후 실행. 실제 호출 시 ANTHROPIC_API_KEY 필요.')
print('Gemini vs Anthropic 차이: 호출 코드 동일, 캐싱 메커니즘만 다름 (CachedContent vs cache_control marker).')

### 2.2 Provider 차이 비교 — 동일 인터페이스, 다른 메커니즘

| 항목 | Gemini | Anthropic |
|---|---|---|
| 호출 시그니처 | `await client.generate(ctx, cache_policy=...)` | 동일 |
| 응답 타입 | `LLMResponse` | 동일 |
| 캐싱 메커니즘 | `CachedContent` 객체 (server resource, name 으로 재호출) | `cache_control: ephemeral` marker (요청마다 송신, server hash 매칭) |
| TTL | `CreateCachedContentConfig(ttl="3600s")` 명시 | ephemeral = 5min default |
| 적중 신호 | `usage.cached_content_token_count > 0` | `usage.cache_read_input_tokens > 0` |
| `force_invalidate=True` | in-memory map 에서 hash 제거 + 새 cache 생성 | marker 미부착 (= `enabled=False` 와 동일 효과) |

→ **도메인 호출자는 차이 모름**. `LLMClient` Protocol + `CachePolicy` 통합 인터페이스가 모두 흡수.

### 2.3 도메인 어댑터 등록 — 새 provider 끼우기 (Open/Closed)

`LLMClient` Protocol 적합 클래스를 만들면 끝 — 베이스 코드 0줄 수정. `runtime_checkable` 로 isinstance 가능 (단 method 존재만 검증, 시그니처는 mypy/ty 위임).

In [ ]:
# 도메인이 직접 어댑터 만들기 (예: 사내 vLLM, 다른 SDK 등)
class MyFakeLLM:
    async def generate(self, ctx, *, cache_policy=None):
        return LLMResponse(
            text='fake response',
            static_hash=get_static_hash(ctx),
            cache_hit=False,
            usage=TokenUsage(input_tokens=10, output_tokens=5),
        )
    def count_tokens(self, text: str) -> int:
        return len(text.split())

fake = MyFakeLLM()
assert isinstance(fake, LLMClient)
print(f'MyFakeLLM 가 LLMClient Protocol 적합: {isinstance(fake, LLMClient)}')

resp = asyncio.run(fake.generate(RenderContext()))
print(f'fake resp: text={resp.text!r} hash={resp.static_hash}')

## 3부: escape hatch + 위험 가드

### 3.1 `CachePolicy` 컨트롤

- `enabled=False` — 캐시 우회
- `force_invalidate=True` — Gemini 는 새 cache 생성 / Anthropic 은 marker 미부착 (= disabled 와 동일 효과)
- `ttl_seconds` — Gemini 만 명시 적용. Anthropic 은 ephemeral 5min default.

In [ ]:
default_policy = CachePolicy()
print(f'default: enabled={default_policy.enabled} ttl={default_policy.ttl_seconds}s force_invalidate={default_policy.force_invalidate}')

disabled = CachePolicy(enabled=False)
print(f'disabled: enabled={disabled.enabled}')

force_inv = CachePolicy(force_invalidate=True)
print(f'force_invalidate: {force_inv.force_invalidate}')

# frozen 검증
from pydantic import ValidationError
try:
    default_policy.enabled = False
except ValidationError as e:
    print(f'\nfrozen 검증 OK — mutation 거부: {type(e).__name__}')

### 3.2 메트릭 observer 등록

**메트릭 계약 (중요)**: `HASH_CHANGE` 는 같은 호출 안에서 후속 `MISS` 와 union 으로 발생하는 *causal annotation*. `HASH_CHANGE + MISS` 합산 시 cache 압력 overcount 위험. `HIT / MISS` 만으로 적중률 계산.

In [ ]:
metrics = CacheMetrics()
events = []
metrics.add_observer(lambda ev, h: events.append((ev.value, h[:8])))

# 실제 호출 없이 emit 시뮬레이션
metrics.emit(CacheEvent.MISS, 'abc12345' + 'x' * 8)
metrics.emit(CacheEvent.HIT, 'abc12345' + 'x' * 8)
metrics.emit(CacheEvent.HASH_CHANGE, 'def00000' + 'x' * 8)
metrics.emit(CacheEvent.MISS, 'def00000' + 'x' * 8)

print(f'events 시퀀스: {events}')
print(f'\nstats: {metrics.stats()}')
print(f'주의: HASH_CHANGE 1 + MISS 2 = 합산 시 3 인데 실제 cache pressure 는 MISS 2 회만.')

### 3.3 R-7 — Anthropic `cache_control` marker 위치 회귀

marker 가 system 메시지의 마지막 text block 에 정확히 부착돼야 server-side hash 매칭. 베이스 코드는 single-point fix (single dict 의 `cache_control` 키). 도메인이 직접 marker 만지면 회귀 위험.

In [ ]:
# 위 mock 호출 검증 패턴 (test_anthropic_adapter.py 참조)
# system_block 은 단일 dict 라 marker 위치 흔들릴 일 자체가 없음
system_block_with_cache = {'type': 'text', 'text': '<static prompt>', 'cache_control': {'type': 'ephemeral'}}
system_block_no_cache = {'type': 'text', 'text': '<static prompt>'}

print(f'enabled 시: {system_block_with_cache}')
print(f'disabled 시: {system_block_no_cache}')
print(f'\nmarker 부착 위치: system 의 마지막 (= 단일 element 라 첫 = 마지막)')

## Cleanup

본 노트북은 `tests/conftest.py` 의 `restore_registry` autouse fixture 와 무관 (Jupyter 는 pytest 와 별개). 다음 cell 실행 시 잔존 metrics / 도메인 register 격리하려면 fresh `CacheMetrics()` 인스턴스 + Phase 1 `registry` snapshot/restore 패턴 직접 적용.

In [ ]:
from best_agent_base.prompts.registry import registry

snapshot = dict(registry._sections)
print(f'registry snapshot: {len(snapshot)} sections')
# 필요 시 후속 셀에서:
# registry._sections.clear()
# registry._sections.update(snapshot)

## 다른 모듈과의 연계

본 모듈의 캐시 적중률은 **시스템 프롬프트 모듈** 의 `get_static_hash(ctx)` 결정성에 100% 의존. 도메인이 동적 섹션 (`static=False`) register 하면 정적부에 영향 X → 캐시 적중 안정. 정적 섹션을 override 하면 정적부 변동 → `HASH_CHANGE` 발생.

In [ ]:
ctx = RenderContext()
h1 = get_static_hash(ctx)
h2 = get_static_hash(ctx)
h3 = get_static_hash(ctx)
print(f'동일 ctx 3 회: h1={h1} h2={h2} h3={h3}')
print(f'동일성 보장: {h1 == h2 == h3}')
print(f'→ 캐시 키 결정성 (NFR-3) source of truth')

## 실습

### 실습 1: `MyFakeLLM` 에 `cache_hit=True` 응답 케이스 추가

두 번째 호출 시 `cache_hit=True` + `usage.cached_tokens > 0` 반환하도록 변경. 도메인 어댑터의 자체 캐시 시뮬레이션 패턴.

In [ ]:
# TODO: MyFakeLLM 을 확장해 인스턴스 단위 cache_map 추가
# class MyFakeLLM_v2:
#     def __init__(self):
#         self._seen = set()
#     async def generate(self, ctx, *, cache_policy=None):
#         h = get_static_hash(ctx)
#         hit = h in self._seen
#         self._seen.add(h)
#         return LLMResponse(text='...', static_hash=h, cache_hit=hit, usage=TokenUsage(...))

### 실습 2: 메트릭 observer 로 적중률 (%) 계산

`CacheMetrics.stats()` 의 `hit / (hit + miss)` 계산 함수 작성. `HASH_CHANGE` 는 합산 X (계약 §3.2).

In [ ]:
# TODO: hit_rate(metrics: CacheMetrics) -> float 작성
# def hit_rate(metrics: CacheMetrics) -> float:
#     stats = metrics.stats()
#     hits = stats.get('hit', 0)
#     misses = stats.get('miss', 0)
#     ...
#     return hits / (hits + misses) if (hits + misses) > 0 else 0.0

### 실습 3: Anthropic 의 `force_invalidate=True` 가 disabled 와 동일 효과인 이유

`anthropic.py:generate()` 코드를 읽고, `cache_policy.enabled=True` + `force_invalidate=True` 조합에서 실제 SDK 호출의 `system` 파라미터에 `cache_control` 키가 어떻게 되는지 추적.

In [ ]:
# TODO: 다음 코드 라인 (best_agent_base/llm/anthropic.py L70 근처) 분석:
# system_block: dict = {'type': 'text', 'text': static_text}
# if policy.enabled and not policy.force_invalidate:
#     system_block['cache_control'] = {'type': 'ephemeral'}
#
# Q: enabled=True + force_invalidate=True 일 때 cache_control 키 추가됨? (Hint: not True == False)

### 실습 4: 도메인 RenderContext 확장으로 동적 컨텍스트 추가

도메인이 자기 `MyContext(RenderContext)` 만들고, `static=False` PromptSection register 해서 매 호출마다 다른 dynamic 영역 주입. `static_hash` 가 변하지 않음 (정적 섹션은 그대로) 을 검증.

In [ ]:
# TODO:
# 1. MyContext(RenderContext) 정의 — 필드 1개 (예: user_id: str)
# 2. DynamicUserSection PromptSection 정의 — static=False, render(ctx) -> f'User: {ctx.user_id}'
# 3. registry.register('DynamicUser', DynamicUserSection())
# 4. 다른 user_id 두 번 호출 → static_hash 동일, dynamic_text 다름 검증
# 5. cleanup: registry 에서 DynamicUserSection 제거